In [10]:
pip install google-api-python-client

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install google-auth google-auth-httplib2 google-auth-oauthlib

Note: you may need to restart the kernel to use updated packages.


In [12]:
from googleapiclient.discovery import build
import csv
from datetime import datetime, timezone
import time

In [ ]:
# 설정
API_KEY = ""
OUTPUT_FILE = "youtube_public_playlists_full_last2.csv"
MAX_COMMENTS_PER_VIDEO = 500

# 수집 대상 공개 채널 (채널 ID)
CHANNELS = {
    # 채널이름 : 채널아이디
    "Seta Kim": ""
}

# YouTube API 빌드
youtube = build("youtube", "v3", developerKey=API_KEY)

In [14]:
# 공개 채널 재생목록 가져오기
def fetch_playlists(channel_id):
    playlists = []
    next_page_token = None
    while True:
        res = youtube.playlists().list(
            part="id,snippet",
            channelId=channel_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        for pl in res.get("items", []):
            playlists.append({
                "playlist_id": pl["id"],
                "title": pl["snippet"]["title"]
            })
        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break
    return playlists

In [15]:
# 재생목록 안 영상 가져오기
def fetch_playlist_videos(playlist_id):
    videos = []
    next_page_token = None
    while True:
        res = youtube.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        for item in res.get("items", []):
            videos.append({
                "video_id": item["snippet"]["resourceId"]["videoId"],
                "title": item["snippet"]["title"],
                "description": item["snippet"]["description"],
                "publish_time": item["snippet"]["publishedAt"]
            })
        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break
    return videos

In [16]:
# 영상 통계 조회
def fetch_video_stats(video_id):
    resp = youtube.videos().list(
        part="snippet,statistics",
        id=video_id
    ).execute()

    if not resp["items"]:
        return None

    item = resp["items"][0]

    return {
        "upload_time": item["snippet"]["publishedAt"], # :point_left: 이게 진짜 업로드 시각
        "viewCount": item["statistics"].get("viewCount","0"),
        "likeCount": item["statistics"].get("likeCount","0"),
        "commentCount": item["statistics"].get("commentCount","0")
    }

In [17]:
# 댓글 수집 + 좋아요
def fetch_comments(video_id):
    all_comments = []
    next_page_token = None

    while True:
        res = youtube.commentThreads().list(
            part="snippet,replies",
            videoId=video_id,
            maxResults=100,
            textFormat="plainText",
            pageToken=next_page_token
        ).execute()

        for thread in res.get("items", []):
            top = thread["snippet"]["topLevelComment"]["snippet"]
            all_comments.append({
                "text": top["textDisplay"],
                "likeCount": top.get("likeCount", 0)
            })

            if "replies" in thread:
                for reply in thread["replies"]["comments"]:
                    r = reply["snippet"]
                    all_comments.append({
                        "text": r["textDisplay"],
                        "likeCount": r.get("likeCount", 0)
                    })

        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break

    # 좋아요 내림차순 정렬 후 최대 500개만
    all_comments.sort(key=lambda x: x["likeCount"], reverse=True)
    return all_comments[:500]

In [18]:
# 영상 + 댓글 + 통계 처리
def process_video(video, playlist_title):
    video_id = video["video_id"]
    publish_utc = datetime.fromisoformat(video["publish_time"].replace("Z","+00:00")).strftime("%Y-%m-%d %H:%M:%S")

    stats = fetch_video_stats(video_id)
    comments = fetch_comments(video_id)

    rows = []
    if not comments:
        rows.append([
            playlist_title, video_id, video["title"], video["description"], stats["upload_time"],
            stats["viewCount"], stats["likeCount"], stats["commentCount"],
            "", ""])
    else:
        for c in comments:
            rows.append([
                playlist_title, video_id, video["title"], video["description"], stats["upload_time"],
                stats["viewCount"], stats["likeCount"], stats["commentCount"],
                c["text"], c["likeCount"]
            ])
    return rows

In [19]:
# 메인 수집
def collect_youtube_public_playlists():
    total_processed = 0
    start_time = time.time()

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "playlist_title",
            "video_id",
            "title",
            "description",
            "publish_time_utc",
            "viewCount",
            "likeCount",
            "commentCount",
            "comment_text",
            "comment_likeCount"
        ])

        for channel_name, channel_id in CHANNELS.items():
            print(f"\n===== 채널 수집 시작: {channel_name} =====")
            playlists = fetch_playlists(channel_id)
            print(f"총 {len(playlists)}개의 재생목록 발견")

            for pl in playlists:
                playlist_title = pl["title"]
                print(f"\n----- 재생목록 수집 시작: {playlist_title} -----")
                videos = fetch_playlist_videos(pl["playlist_id"])
                print(f"총 {len(videos)}개의 영상 발견")

                for video in videos:
                    rows = process_video(video, playlist_title)
                    for row in rows:
                        writer.writerow(row)
                    total_processed += 1

                    # 진행률 및 ETA
                    progress_ratio = total_processed / max(len(videos),1)
                    elapsed_time = time.time() - start_time
                    avg_time = elapsed_time / max(total_processed,1)
                    remaining_time = avg_time * (len(videos) - total_processed)

                    if total_processed % 5 == 0:
                        print(
                            f"[{playlist_title}] 저장 중 영상 ID: {rows[0][1]} | "
                            f"게시글 UTC: {rows[0][4]} | "
                            f"진행률: {progress_ratio*100:.2f}% | "
                            f"총 처리: {total_processed}개 | "
                            f"평균: {avg_time:.2f}초/개 | "
                            f"예상 남은 시간: {remaining_time/60:.1f}분"
                        )

                print(f"----- 재생목록 수집 완료: {playlist_title} -----")

            print(f"===== 채널 수집 완료: {channel_name} =====")

    print("\n===== 전체 YouTube 공개 재생목록 수집 완료 =====")


In [20]:
# 실행
collect_youtube_public_playlists()


===== 채널 수집 시작: Seta Kim =====
총 1개의 재생목록 발견

----- 재생목록 수집 시작: Test -----
총 51개의 영상 발견
[Test] 저장 중 영상 ID: YbwJUvc8Hnc | 게시글 UTC: 2025-10-01T12:28:53Z | 진행률: 9.80% | 총 처리: 5개 | 평균: 1.74초/개 | 예상 남은 시간: 1.3분
[Test] 저장 중 영상 ID: Y0OrNb7SJx0 | 게시글 UTC: 2025-05-29T11:50:50Z | 진행률: 19.61% | 총 처리: 10개 | 평균: 1.91초/개 | 예상 남은 시간: 1.3분
[Test] 저장 중 영상 ID: q9Upcp9bPjM | 게시글 UTC: 2025-06-18T11:12:55Z | 진행률: 29.41% | 총 처리: 15개 | 평균: 2.15초/개 | 예상 남은 시간: 1.3분
[Test] 저장 중 영상 ID: 7cl0BPVBd0A | 게시글 UTC: 2025-11-13T13:23:20Z | 진행률: 39.22% | 총 처리: 20개 | 평균: 1.75초/개 | 예상 남은 시간: 0.9분
[Test] 저장 중 영상 ID: mYycNIlkIrc | 게시글 UTC: 2025-11-13T22:54:43Z | 진행률: 49.02% | 총 처리: 25개 | 평균: 1.52초/개 | 예상 남은 시간: 0.7분
[Test] 저장 중 영상 ID: XJfOG5sdwQY | 게시글 UTC: 2023-10-19T05:12:08Z | 진행률: 58.82% | 총 처리: 30개 | 평균: 1.40초/개 | 예상 남은 시간: 0.5분
[Test] 저장 중 영상 ID: LBrGVnak54A | 게시글 UTC: 2023-11-16T08:06:09Z | 진행률: 68.63% | 총 처리: 35개 | 평균: 1.37초/개 | 예상 남은 시간: 0.4분
[Test] 저장 중 영상 ID: Xhzd97efUDw | 게시글 UTC: 2025-10-31T08:01:58Z | 진행률: 78.

In [32]:
import pandas as pd

translated_csv_path = "youtube_public_playlists_full_last2.csv"
df= pd.read_csv(translated_csv_path, encoding="utf-8-sig")
df

,playlist_title,video_id,title,description,publish_time_utc,viewCount,likeCount,commentCount,comment_text,comment_likeCount,source
0,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,nc가 회사 명운을 걸고 오픈한 게임이 두시간만에 이룩한 업적들\n1.대기열은 기본...,53,국내
1,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,다터져서 아무도 못들어가고있는데 네번째 브랜드무비 이지랄ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,36,국내
2,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,남준이형 일어나봐...,31,국내
3,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,아니 사람 이렇게 몰릴것 대비 안한 엔씨 나와라 .. \n시작하자마자 아무도 접속 ...,27,국내
4,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,영혼의 서 큐나 구매 아니고 현금 패키지 구매 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,26,국내
...,...,...,...,...,...,...,...,...,...,...,...
7573,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,ㅄ같은 마사지를 보았습니다,0,국내
7574,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,0,국내
7575,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,lets gooooooo plz don't make same mistakes yo...,0,국내
7576,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,클베 언제인지 아시나요?,0,국내


In [ ]:
""" 
# 국내 표시 없을 경우 진행 코드

# source 컬럼 추가 (전부 "국내")
df["source"] = "국내"

# 다시 저장 (덮어쓰기 or 새 파일)
df.to_csv(file_path, index=False, encoding="utf-8-sig")
"""

In [36]:
pip install langdetect

  Using cached langdetect-1.0.9.tar.gz (981 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993364 sha256=5c32521cbd3f1798b2d298fbe543faf9853287cc2e23462c5b130021d4f5ddfe
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\eb\87\25\2dddf1c94e1786054e25022ec5530bfed52bad86d882999c48
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.


In [37]:
# 유튜브에 다양한 언어 댓글 이슈가 있어 언어 인식 먼저 진행
from langdetect import detect, DetectorFactory

In [38]:
# 결과 안정화
DetectorFactory.seed = 0

# 파일 불러오기
file_path = "youtube_public_playlists_full_last2.csv"
df = pd.read_csv(file_path)

# 언어 감지 함수
def detect_language(text):
    try:
        if pd.isna(text) or str(text).strip() == "":
            return "unknown"
        
        text = str(text)

        # 한글 포함 시 무조건 한국어
        if any('\uac00' <= c <= '\ud7a3' for c in text):
            return "ko"

        return detect(text)
    except:
        return "unknown"

# language 컬럼 추가
df["language"] = df["comment_text"].apply(detect_language)

# 저장
df.to_csv(file_path, index=False, encoding="utf-8-sig")

print("language 컬럼 추가 완료")

language 컬럼 추가 완료


In [39]:
df

,playlist_title,video_id,title,description,publish_time_utc,viewCount,likeCount,commentCount,comment_text,comment_likeCount,source,language
0,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,nc가 회사 명운을 걸고 오픈한 게임이 두시간만에 이룩한 업적들\n1.대기열은 기본...,53,국내,ko
1,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,다터져서 아무도 못들어가고있는데 네번째 브랜드무비 이지랄ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,36,국내,ko
2,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,남준이형 일어나봐...,31,국내,ko
3,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,아니 사람 이렇게 몰릴것 대비 안한 엔씨 나와라 .. \n시작하자마자 아무도 접속 ...,27,국내,ko
4,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,영혼의 서 큐나 구매 아니고 현금 패키지 구매 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,26,국내,ko
...,...,...,...,...,...,...,...,...,...,...,...,...
7573,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,ㅄ같은 마사지를 보았습니다,0,국내,ko
7574,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,0,국내,ko
7575,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,lets gooooooo plz don't make same mistakes yo...,0,국내,et
7576,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,클베 언제인지 아시나요?,0,국내,ko


In [50]:
# comment_text를 문자열로 강제 변환 + NaN 처리
df["comment_text"] = df["comment_text"].fillna("").astype(str)

problem_data = df[
    (df["comment_text"].str.strip() == "") |
    (df["language"] == "unknown")
]

print("문제 데이터 수:", len(problem_data))
problem_data

문제 데이터 수: 101


,playlist_title,video_id,title,description,publish_time_utc,viewCount,likeCount,commentCount,comment_text,comment_likeCount,source,language
98,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,<3,1,국내,unknown
177,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,🤏,0,국내,unknown
531,Test,8u5VS5HHsxw,🎥 AION2 론칭 전 깜짝 라이브 방송 하이라이트,데바 여러분이 많이 궁금해하셨던\n📌 AION2의 조작법 \n📌 검성/수호성/마도성...,2025-11-14T04:17:30Z,51955,536,209,,0,국내,unknown
641,Test,EI6qhsPIcd0,[AION2] G-STAR 2025 트레일러,"아트레이아의 하늘이 다시 열린다 \n이제, 당신이 날아오를 시간 \n2025. 11...",2025-11-13T03:30:21Z,2386160,1269,380,ㄹㅇㅋㅋㅋㅋ,3,국내,unknown
678,Test,EI6qhsPIcd0,[AION2] G-STAR 2025 트레일러,"아트레이아의 하늘이 다시 열린다 \n이제, 당신이 날아오를 시간 \n2025. 11...",2025-11-13T03:30:21Z,2386160,1269,380,1:21,1,국내,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...
7473,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,😂😂😂😂😂,0,국내,unknown
7499,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,ㄱㄱㄱ,0,국내,unknown
7504,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,ㄱㅈ ㅇㅂㄱㄹㅇㅅ,0,국내,unknown
7570,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ,0,국내,unknown


In [52]:
# 컬럼 강제 정리
df["comment_text"] = df["comment_text"].fillna("").astype(str)
df["language"] = df["language"].fillna("unknown").astype(str)

# 조건을 각각 따로 만들기 (중요)
cond_empty = df["comment_text"].str.strip().eq("")
cond_unknown = df["language"].eq("unknown")

# 문제 데이터
problem_idx = cond_empty | cond_unknown

# 정상 데이터
clean_df = df.loc[~problem_idx].copy()

In [53]:
# 101개 데이터 제거 후 다시 저장 (덮어쓰기 or 새 파일)

# 저장
clean_df.to_csv("youtube_public_playlists_full_last2.csv", index=False, encoding="utf-8-sig")

In [54]:
clean_df

,playlist_title,video_id,title,description,publish_time_utc,viewCount,likeCount,commentCount,comment_text,comment_likeCount,source,language
0,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,nc가 회사 명운을 걸고 오픈한 게임이 두시간만에 이룩한 업적들\n1.대기열은 기본...,53,국내,ko
1,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,다터져서 아무도 못들어가고있는데 네번째 브랜드무비 이지랄ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,36,국내,ko
2,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,남준이형 일어나봐...,31,국내,ko
3,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,아니 사람 이렇게 몰릴것 대비 안한 엔씨 나와라 .. \n시작하자마자 아무도 접속 ...,27,국내,ko
4,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,영혼의 서 큐나 구매 아니고 현금 패키지 구매 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,26,국내,ko
...,...,...,...,...,...,...,...,...,...,...,...,...
7573,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,ㅄ같은 마사지를 보았습니다,0,국내,ko
7574,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,0,국내,ko
7575,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,lets gooooooo plz don't make same mistakes yo...,0,국내,et
7576,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,클베 언제인지 아시나요?,0,국내,ko


In [55]:
# 기존 source 컬럼 삭제
if "source" in clean_df.columns:
    clean_df = clean_df.drop(columns=["source"])

# region 컬럼 추가
clean_df["region"] = "국내"

# source 컬럼 새로 추가
clean_df["source"] = "youtube"

# 저장
clean_df.to_csv("youtube_public_playlists_full_last2.csv", index=False, encoding="utf-8-sig")

print("컬럼 정리 완료")

컬럼 정리 완료


In [57]:
clean_df

,playlist_title,video_id,title,description,publish_time_utc,viewCount,likeCount,commentCount,comment_text,comment_likeCount,language,region,source
0,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,nc가 회사 명운을 걸고 오픈한 게임이 두시간만에 이룩한 업적들\n1.대기열은 기본...,53,ko,국내,youtube
1,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,다터져서 아무도 못들어가고있는데 네번째 브랜드무비 이지랄ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,36,ko,국내,youtube
2,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,남준이형 일어나봐...,31,ko,국내,youtube
3,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,아니 사람 이렇게 몰릴것 대비 안한 엔씨 나와라 .. \n시작하자마자 아무도 접속 ...,27,ko,국내,youtube
4,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,영혼의 서 큐나 구매 아니고 현금 패키지 구매 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,26,ko,국내,youtube
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7573,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,ㅄ같은 마사지를 보았습니다,0,ko,국내,youtube
7574,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,0,ko,국내,youtube
7575,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,lets gooooooo plz don't make same mistakes yo...,0,et,국내,youtube
7576,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,클베 언제인지 아시나요?,0,ko,국내,youtube


In [59]:
import re

def contains_link(text: str) -> bool:
    if pd.isna(text):
        return False
    return bool(re.search(r"http[s]?://\S+", str(text)))

def replace_aion(text: str) -> str:
    if pd.isna(text):
        return text
    return re.sub(r"AION", "아이온", str(text), flags=re.IGNORECASE)

# 링크가 있는 행 삭제
clean_df = clean_df[~(clean_df["title"].apply(contains_link) | clean_df["comment_text"].apply(contains_link))].copy()

In [63]:
# title 컬럼 이름 변경 → selftext
clean_df.rename(columns={"comment_text":"selftext"}, inplace=True)

In [65]:
# 저장
clean_df.to_csv("youtube_public_playlists_full_last2.csv", index=False, encoding="utf-8-sig")

In [64]:
clean_df

,playlist_title,video_id,title,description,publish_time_utc,viewCount,likeCount,commentCount,selftext,comment_likeCount,language,region,source
0,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,nc가 회사 명운을 걸고 오픈한 게임이 두시간만에 이룩한 업적들\n1.대기열은 기본...,53,ko,국내,youtube
1,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,다터져서 아무도 못들어가고있는데 네번째 브랜드무비 이지랄ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,36,ko,국내,youtube
2,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,남준이형 일어나봐...,31,ko,국내,youtube
3,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,아니 사람 이렇게 몰릴것 대비 안한 엔씨 나와라 .. \n시작하자마자 아무도 접속 ...,27,ko,국내,youtube
4,Test,oSbl_ih25IM,[AION2] 네 번째 브랜드 무비,아트레이아가 기다려 온 \n운명의 데바\n바로 당신이기를 \nAION2 [GRAND...,2025-11-18T15:00:19Z,19467254,275,382,영혼의 서 큐나 구매 아니고 현금 패키지 구매 ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ...,26,ko,국내,youtube
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7573,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,ㅄ같은 마사지를 보았습니다,0,ko,국내,youtube
7574,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,0,ko,국내,youtube
7575,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,lets gooooooo plz don't make same mistakes yo...,0,et,국내,youtube
7576,Test,ZvlYP_eu6FQ,[로스트아크 모바일] CLOSED BETA TEST - 트레일러 ㅣ4K UHD,로스트아크 모바일 CLOSED BETA TEST 모집 중!\n\n■ CBT 모집 기...,2025-10-17T05:01:08Z,1456304,852,459,클베 언제인지 아시나요?,0,ko,국내,youtube
